# Phase 2: Discovery Engine

This notebook covers:
1. **K-Means Clustering**: Behavioral customer segmentation
2. **Agglomerative Hierarchical Clustering**: Dendrogram visualization and distance cut-offs
3. **DBSCAN Density Clustering**: Fraudulent transaction and outlier isolation (Epsilon & MinPts)
4. **Dimensionality Reduction**: Principal Component Analysis (PCA) & t-SNE 2D Manifolds

In [ ]:
from mlo.data.generator import generate_customer_dataset
from mlo.data.encoder import CategoricalEncoder
from mlo.data.preprocessor import DataPreprocessor
from mlo.unsupervised.clustering import CustomerKMeans, CustomerHierarchicalClustering
from mlo.unsupervised.density import CustomerDBSCAN
from mlo.unsupervised.dim_reduction import CustomerPCA, CustomerTSNE
from mlo.utils.visualization import plot_cluster_scatter
import matplotlib.pyplot as plt

## 1. Load Data & Prepare Scaled Features

In [ ]:
df = generate_customer_dataset(n_samples=500, random_state=42, anomaly_ratio=0.06)
encoder = CategoricalEncoder()
df_encoded = encoder.fit_transform(df)
X = df_encoded[encoder.feature_columns_].values
preproc = DataPreprocessor(scaler_type="standard")
X_scaled = preproc.fit_transform(X)

## 2. K-Means & Agglomerative Hierarchical Clustering

In [ ]:
kmeans = CustomerKMeans(n_clusters=3, random_state=42).fit(X_scaled)
km_labels = kmeans.predict(X_scaled)
print(f"K-Means Inertia: {kmeans.inertia:.2f}")

hier = CustomerHierarchicalClustering(n_clusters=3, linkage_method="ward")
hier.plot_dendrogram(X_scaled[:40], max_d=6.5)

## 3. DBSCAN Anomaly & Fraud Isolation

In [ ]:
dbscan = CustomerDBSCAN(eps=1.2, min_samples=4)
db_labels = dbscan.fit_predict(X_scaled)
stats = dbscan.get_cluster_stats()
print(f"DBSCAN Clusters: {stats['n_clusters']} | Outliers / Fraud flagged: {stats['n_noise_points']} ({stats['noise_ratio']*100:.1f}%)")

## 4. Dimensionality Reduction (PCA & t-SNE)

In [ ]:
pca = CustomerPCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA Explained Variance: {pca.total_variance_explained*100:.2f}%")

tsne = CustomerTSNE(n_components=2, perplexity=25.0, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)
plot_cluster_scatter(X_tsne[:, 0], X_tsne[:, 1], km_labels, title="Customer Segments via t-SNE Projection")